# Notebook 07 - MAPER Orchestration (LangGraph)

**Purpose**: Wire all four agents into a single LangGraph `StateGraph` orchestrating the full MAPER pipeline end-to-end.

**Agent flow**: Orchestrator -> Profile Agent -> Hybrid Retrieval Agent (dense+BM25+CF+graph fusion, profile rerank) -> Critic-Generator Agent (verify+generate) -> Orchestrator re-routes if trust < 0.8 -> ranked papers + verified rationales.

**Technology**: `langgraph` 0.2, conditional edges, shared state.

**Outputs**: `FinalRecommendation` objects with hallucination scores, trust scores, and grounded rationales.

In [1]:
import os
for f in ["/kaggle/working/ablation_checkpoint.json", "/kaggle/working/ablation_results.json"]:
        if os.path.exists(f):
                    os.remove(f)
                    print("removed:", f)
        else:
            print("not found (already clean):", f)
    

removed: /kaggle/working/ablation_checkpoint.json
removed: /kaggle/working/ablation_results.json


In [2]:
# == Dependencies + environment ==============================================
import json, os, re, time, math, pickle, warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Literal, Optional, TypedDict
import numpy as np
import pandas as pd
import requests
warnings.filterwarnings("ignore")

# NOTE: plain os.path.exists("/kaggle") is not reliable on Windows -- it matches
# case-insensitively against any local "kaggle"-named folder on the current drive root.
# Require the actual Kaggle run-type env var, which only a real Kaggle kernel sets.
IS_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None
OUT_DIR   = Path("/kaggle/working") if IS_KAGGLE else Path("../data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def _find(name):
    for root in ["/kaggle/input", "/kaggle/working", "..", "."]:
        r = Path(root)
        if r.exists():
            hits = list(r.rglob(name))
            if hits:
                return hits[0]
    return None

# Keys: Kaggle Secrets first, then env. Project standard = OpenRouter (NOT OpenAI).
try:
    from kaggle_secrets import UserSecretsClient
    _sec = UserSecretsClient()
    def _secret(k):
        try:
            return _sec.get_secret(k)
        except Exception:
            return ""
except Exception:
    def _secret(k):
        return os.getenv(k, "")

OPENROUTER_API_KEY = _secret("OPENROUTER_API_KEY") or os.getenv("OPENROUTER_API_KEY", "")
S2_API_KEY         = _secret("SEMANTIC_SCHOLAR_API_KEY") or os.getenv("SEMANTIC_SCHOLAR_API_KEY", "")
S2_BASE    = "https://api.semanticscholar.org/graph/v1"
HEADERS_S2 = {"x-api-key": S2_API_KEY} if S2_API_KEY else {}

OPENROUTER_BASE     = "https://openrouter.ai/api/v1"
GEN_MODEL           = "meta-llama/llama-3.3-70b-instruct"
OPENROUTER_PROVIDER = "Novita"

# Set True to call OpenRouter for real rationale generation (spends credit).
# Default False uses a deterministic stub so the full graph-grounded pipeline runs FREE.
RUN_LLM_GEN = True

CITATION_TRUST_THRESHOLD = 0.80
TOP_K = 10

try:
    from langgraph.graph import StateGraph, END
    LANGGRAPH_AVAILABLE = True
    print("LangGraph available.")
except ImportError:
    LANGGRAPH_AVAILABLE = False
    print("LangGraph not installed (pip install langgraph) - using sequential fallback.")

print("OpenRouter key:", bool(OPENROUTER_API_KEY), "| S2 key:", bool(S2_API_KEY),
      "| IS_KAGGLE:", IS_KAGGLE)

LangGraph available.
OpenRouter key: True | S2 key: True | IS_KAGGLE: True


In [3]:
# == Load artifacts (same dataset paths as nb04/nb05; reuse existing GTE embeddings) ==
import torch
import importlib.util as _ilu, subprocess as _sp, sys as _sys
for _pkg in ["rank-bm25", "sentence-transformers"]:
    if _ilu.find_spec(_pkg.replace("-", "_")) is None:
        _sp.run([_sys.executable, "-m", "pip", "install", "-q", _pkg], check=False)
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---- Papers metadata: JSON at reapers2025 (same as nb04/nb05), _find fallback
_PAPERS_JSON = Path("/kaggle/input/datasets/reapers2025/paper-metadata/paper_metadata.json")
if _PAPERS_JSON.exists():
    papers_df = pd.read_json(_PAPERS_JSON)
else:
    _pp = _find("paper_metadata.json") or _find("papers_metadata.csv")
    assert _pp is not None, "papers metadata not found (paper_metadata.json / papers_metadata.csv)"
    papers_df = pd.read_json(_pp) if str(_pp).endswith(".json") else pd.read_csv(_pp)
id_col = next((c for c in ["arxiv_id", "paper_id", "paperId", "id"] if c in papers_df.columns), None)
_pdf = papers_df.drop_duplicates(subset=[id_col]).set_index(id_col) if id_col else papers_df

# ---- Embeddings: e5 if present, else existing GTE-Large (query encoder must match)
_e5_emb, _e5_map = _find("paper_embeddings_e5.npy"), _find("paper_id_mapping_e5.json")
if _e5_emb and _e5_map:
    emb_matrix = np.load(_e5_emb); id_mapping = json.load(open(_e5_map, encoding="utf-8"))
    EMBED_MODEL, QUERY_PREFIX = "intfloat/e5-large-v2", "query: "
else:
    _EMB_HARD = Path("/kaggle/input/datasets/reapers2025/acl-embeddings/paper_embeddings_full.npy")
    _MAP_HARD = Path("/kaggle/input/datasets/reapers2025/acl-embeddings/paper_id_mapping.json")
    _emb_p = _EMB_HARD if _EMB_HARD.exists() else _find("paper_embeddings_full.npy")
    _map_p = _MAP_HARD if _MAP_HARD.exists() else _find("paper_id_mapping.json")
    assert _emb_p and _map_p, "GTE embeddings not found (paper_embeddings_full.npy / paper_id_mapping.json)"
    emb_matrix = np.load(_emb_p); id_mapping = json.load(open(_map_p, encoding="utf-8"))
    EMBED_MODEL, QUERY_PREFIX = "thenlper/gte-large", ""
emb_matrix  = emb_matrix.astype(np.float32)
_idx_to_pid = {int(v): k for k, v in id_mapping.items()}
_doc_norms  = np.linalg.norm(emb_matrix, axis=1) + 1e-9
embed_model = SentenceTransformer(EMBED_MODEL, device=DEVICE)

# ---- BM25: load nb05's index if present, else build (cached)
# BUG FIXED: `_bm = Path(...); if _bm:` was always truthy (a Path object is never
# falsy, even if the file doesn't exist) -- so if bm25_index.pkl wasn't attached at
# EXACTLY that hardcoded path, this crashed with FileNotFoundError instead of falling
# through to the "build it fresh" branch the code clearly intends. Now checks
# .exists() and falls back to _find(), then to building fresh, same pattern already
# fixed elsewhere in this project (01_citecheck, 08_dpo_pipeline).
_BM_HARD = Path("/kaggle/input/datasets/joytuntonny/rationale-subset/bm25_index.pkl")
_bm = _BM_HARD if _BM_HARD.exists() else _find("bm25_index.pkl")
if _bm:
    _d = pickle.load(open(_bm, "rb")); bm25_index = _d["bm25"]; bm25_doc_ids = _d["doc_ids"]
    print("loaded BM25 index:", _bm)
else:
    print("bm25_index.pkl not found -- building BM25 over corpus (one-time)...")
    _toks, bm25_doc_ids = [], []
    for _, row in papers_df.iterrows():
        _toks.append((str(row.get("title", "") or "") + " " + str(row.get("abstract", "") or "")).lower().split())
        bm25_doc_ids.append(str(row[id_col]))
    bm25_index = BM25Okapi(_toks)
    with open(OUT_DIR / "bm25_index.pkl", "wb") as f:
        pickle.dump({"bm25": bm25_index, "doc_ids": bm25_doc_ids}, f)
_bm25_pos = {pid: i for i, pid in enumerate(bm25_doc_ids)}   # pid -> row index for per-candidate lookup

# ---- Optional artifacts: graph scores + user profiles (pinned, _find fallback)
# Same Path-truthy bug fixed here for graph_scores.json (was `if _gp else {}`, always
# truthy -- crashed instead of degrading to {} when the file wasn't at the exact
# hardcoded path).
_GP_HARD = Path("/kaggle/input/datasets/joytuntonny/rationale-subset/graph_scores.json")
_gp = _GP_HARD if _GP_HARD.exists() else _find("graph_scores.json")
graph_scores = json.load(open(_gp, encoding="utf-8")) if _gp else {}
if not graph_scores:
    print("WARNING: graph_scores.json NOT found - running with graph_score=0 for all candidates.")

_UP_HARD = Path("/kaggle/input/datasets/joytuntonny/rationale-subset/user_profiles.json")
_up = _UP_HARD if _UP_HARD.exists() else _find("user_profiles.json")
user_profiles_store = json.load(open(_up, encoding="utf-8")) if _up else {}
if not user_profiles_store:
    print("WARNING: user_profiles.json NOT found - running WITHOUT personalization.")

# ---- Collaborative-filtering signal: SHARED ALS model trained once in nb10 -----------------
# Same shared artifact nb05 loads -- one trained model, reused everywhere. item_factors rows
# are aligned to id_mapping's insertion order (verified in nb10), so cf_item_factors[
# id_mapping[pid]] is that paper's latent vector directly. user_factors rows are aligned to
# als_user_ids.json; a user_id absent from that list is CF-cold-start for this notebook too.
_cf_items_p = _find("als_item_factors.npy")
_cf_users_p = _find("als_user_factors.npy")
_cf_ids_p   = _find("als_user_ids.json")
if _cf_items_p and _cf_users_p and _cf_ids_p:
    cf_item_factors = np.load(_cf_items_p)
    cf_user_factors = np.load(_cf_users_p)
    cf_user_row = {u: i for i, u in enumerate(json.load(open(_cf_ids_p, encoding="utf-8")))}
    print(f"loaded shared ALS CF model: items {cf_item_factors.shape}, users {cf_user_factors.shape} "
          f"({len(cf_user_row)} users with a CF vector)")
else:
    cf_item_factors, cf_user_factors, cf_user_row = None, None, {}
    print("WARNING: shared ALS CF artifacts NOT found -- attach nb10's output dataset. "
          "Running WITHOUT the CF signal (retrieval falls back to dense+BM25+graph only).")

print(f"papers {len(papers_df)} | emb {emb_matrix.shape} | encoder {EMBED_MODEL} | "
      f"profiles {len(user_profiles_store)} | graph_scores {len(graph_scores)} | "
      f"cf_users {len(cf_user_row)}")

modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/670M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

... (3 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

loaded BM25 index: /kaggle/input/datasets/joytuntonny/rationale-subset/bm25_index.pkl
loaded shared ALS CF model: items (239618, 64), users (14861, 64) (14861 users with a CF vector)
papers 239618 | emb (239618, 1024) | encoder thenlper/gte-large | profiles 13223 | graph_scores 235636 | cf_users 14861


## 1. Shared State Definition

In [4]:
class MAPERState(TypedDict):
    """Shared state passed between agents in the LangGraph StateGraph."""
    # Input
    user_id: str
    query: str                   # may be enriched by Profile Agent in-place
    query_type: str              # "history_based" | "natural_language" | "hybrid"
    original_query: str          # unmodified user-typed query (preserved throughout)

    # Ablation configuration (see Section "Experimental Setup" of the manuscript):
    #   "a" = single-pass, text-only  (no re-routing loop, no CiteCheck Stage 4)
    #   "b" = multi-stage, text-only  (re-routing loop enabled, no CiteCheck Stage 4)
    #   "c" = multi-stage + CiteCheck (both enabled -- this is the paper's full system)
    verification_config: str

    # Profile Agent output
    user_profile: Dict[str, Any]

    # Retrieval Agent output
    candidates: List[Dict[str, Any]]  # serialized CandidatePaper objects

    # Critic-Generator output
    verification_results: List[Dict[str, Any]]

    # Final output
    final_recommendations: List[Dict[str, Any]]

    # Control flow signals
    retrieval_iteration: int
    max_iterations: int
    low_trust_paper_ids: List[str]  # papers that failed the trust gate
    n_relational_hallucinations: int  # graph-Checker (CiteCheck) flags
    status: str  # "running" | "complete" | "error"
    error_message: str

print("MAPERState TypedDict defined.")


MAPERState TypedDict defined.


## 2. Agent Node Functions

In [5]:
# == CiteCheck (graph-grounded relational verifier) + relation extraction =====
# Stage 4 of the Checker. Same STRUCTURAL logic as notebook 01 verify_relation:
# a claimed paper->paper relation is 'supported' only if the citation graph shows a
# link; 'unsupported' (claimed dependency with NO citation edge) is a RELATIONAL
# hallucination that the text-only Stages 1-3 structurally cannot catch.
_ARXIV_RE = re.compile(r"^\d{4}\.\d{4,5}")
def _s2id(pid):
    pid = str(pid)
    return "ARXIV:" + pid if _ARXIV_RE.match(pid) else pid

def _s2_batch(ids, fields, chunk=100):
    res = {}
    for i in range(0, len(ids), chunk):
        sub = ids[i:i + chunk]
        for attempt in range(5):
            try:
                rr = requests.post(f"{S2_BASE}/paper/batch", headers=HEADERS_S2,
                                   params={"fields": fields}, json={"ids": sub}, timeout=40)
                if rr.status_code == 200:
                    for pid, obj in zip(sub, rr.json()):
                        res[pid] = obj
                    break
                elif rr.status_code == 429:
                    time.sleep(2 * (attempt + 1))
                else:
                    time.sleep(1); break
            except Exception:
                time.sleep(1)
        time.sleep(0.4)
    return res

class CiteCheckVerifier:
    # Calibrated on nb2 human gold (5-fold CV, see 01_citecheck cell 4): tau=1 beat the
    # uncalibrated default of 3 out-of-sample (F1 0.442 vs 0.437). Keep in sync with nb01.
    COCITE_MIN = 1
    def __init__(self, cache_path):
        self.cache_path = Path(cache_path)
        self.cache = {}
        if self.cache_path.exists():
            try: self.cache = json.loads(self.cache_path.read_text(encoding="utf-8"))
            except Exception: self.cache = {}
        else:
            seed = _find("citecheck_refs_cache.json")   # reuse nb01's graph if attached
            if seed:
                try: self.cache = json.loads(Path(seed).read_text(encoding="utf-8"))
                except Exception: self.cache = {}
        self._index()

    def _index(self):
        self.refs, self.year, self.pid_of = {}, {}, {}
        for raw, e in self.cache.items():
            if e and e.get("pid"):
                self.pid_of[raw] = e["pid"]
                self.year[raw]   = e.get("year")
                self.refs[raw]   = set(e.get("refs") or []) | {e["pid"]}

    def ensure(self, paper_ids):
        ids  = [str(p) for p in paper_ids if p]
        todo = [p for p in dict.fromkeys(ids) if p not in self.cache]
        if todo and HEADERS_S2:
            got = _s2_batch([_s2id(p) for p in todo], "paperId,year,references.paperId", chunk=100)
            for raw in todo:
                obj = got.get(_s2id(raw)) or {}
                self.cache[raw] = {"pid":  obj.get("paperId"),
                                   "year": obj.get("year"),
                                   "refs": [x.get("paperId") for x in (obj.get("references") or []) if x.get("paperId")]}
            try: self.cache_path.write_text(json.dumps(self.cache), encoding="utf-8")
            except Exception: pass
            self._index()

    def cites(self, a, b):
        return bool(self.pid_of.get(b) and a in self.refs and self.pid_of[b] in self.refs[a])

    def shared_refs(self, a, b):
        return len(self.refs.get(a, set()) & self.refs.get(b, set())) if (a in self.refs and b in self.refs) else 0

    def verify_relation(self, a, relation, b):
        if a not in self.refs or b not in self.refs:
            return "unverifiable"
        a_b, b_a = self.cites(a, b), self.cites(b, a)
        sh = self.shared_refs(a, b); ya, yb = self.year.get(a), self.year.get(b)
        if relation in ("builds_on", "uses_method_of", "supersedes"):
            if a_b and (ya is None or yb is None or ya >= yb):
                return "supported"
            if b_a:
                return "supported"
            return "unsupported"
        if relation == "compares":
            return "supported" if (a_b or b_a or sh >= self.COCITE_MIN) else "unsupported"
        if relation == "shares_topic":
            return "supported" if (sh >= self.COCITE_MIN or a_b or b_a) else "unsupported"
        return "unverifiable"

citecheck = CiteCheckVerifier(OUT_DIR / "citecheck_refs_cache.json")
print(f"CiteCheck ready | graph cache: {len(citecheck.cache)} papers | S2:", bool(HEADERS_S2))

# ---- Relational-claim extraction (mirrors benchmark detect_relation) ----------
_REL_CUES = {
    "builds_on":    ["builds on", "build on", "based on", "extends", "extension of",
                     "inspired by", "leverages", "follows from", "draws on", "grounded in"],
    "compares":     ["outperforms", "compared to", "compared with", "in contrast", "unlike",
                     "better than", " versus ", " vs ", "improves over", "baseline"],
    "shares_topic": ["both ", "similarly", "also addresses", "shares ", "same ",
                     "related to", "in line with", "complements"],
    "supersedes":   ["supersedes", "replaces", "improves upon", "succeeds"],
}
def _rel_toks(txt):
    return {w.lower() for w in re.findall(r"[A-Za-z][A-Za-z\-]{2,}", str(txt))}

def _match_titles(claim, cand_papers, k=2):
    """Resolve paper mentions in `claim`. Tries, in order: (1) [N] bracket numbers
    positional into `cand_papers` -- reliable IFF cand_papers is the exact numbered list
    shown to whatever produced the claim (real-LLM path passes the small numbered subset
    actually shown in the prompt; the stub path passes the full candidate list and never
    emits bracket numbers, so this pass is a no-op there). Returns ALL distinct bracket
    matches (not capped at k) so detect_relation can see whether the subject paper is
    among them before deciding how to pair the two endpoints -- capping here would let an
    early match consume the k budget and silently drop a later bracket for the subject
    itself. (2) quoted-title match; (3) token-overlap fallback -- both capped at k, since
    they don't have this same early-cutoff-vs-subject interaction."""
    bracket_found = []
    for m in re.finditer(r"\[(\d+)\]", claim):
        n = int(m.group(1))
        if 1 <= n <= len(cand_papers):
            pid = cand_papers[n - 1].get("paper_id", "")
            if pid and pid not in bracket_found:
                bracket_found.append(pid)
    if bracket_found:
        return bracket_found

    quoted = re.findall(r'["“”]([^"“”]{6,})["“”]', claim)
    claim_tok = _rel_toks(claim)
    found = []
    for p in cand_papers:
        pid = p.get("paper_id", "")
        if not pid or pid in found:
            continue
        t = (p.get("title", "") or "").strip()
        if not t:
            continue
        tl = t.lower()
        hit = any(q.lower()[:40] in tl or tl[:40] in q.lower() for q in quoted)
        if not hit:
            tt = _rel_toks(t)
            if tt and len(tt & claim_tok) / len(tt) >= 0.6:
                hit = True
        if hit:
            found.append(pid)
        if len(found) >= k:
            break
    return found

def detect_relation(claim, cand_papers, subject_id=None):
    """Detect a relational claim and resolve its two paper endpoints.

    BUG FIXED: the rationale (real or stub) almost always refers to the paper the
    rationale is ABOUT implicitly ("this paper...", "It builds on...", or "[1]" when [1]
    IS the subject) rather than by its own quoted title -- so _match_titles only ever
    found the OTHER paper mentioned, and the old code assigned that single match to
    paper_a (ps[0]), leaving paper_b=None and forcing every such claim to verdict
    "unverifiable" regardless of the citation graph. That is why live coverage was ~26%
    even though the graph itself had the answer. Passing subject_id (the candidate whose
    rationale this is) lets us assign the implicit self-reference correctly: if the
    subject IS among the resolved mentions, pair it with the first OTHER distinct
    mention; if the subject is NOT mentioned at all, either the claim is genuinely about
    two other papers (2+ found) or the subject is implicit (1 found, e.g. "this paper...
    builds on X") -- both handled below.
    """
    cl = " " + claim.lower() + " "
    relation = next((rel for rel, cues in _REL_CUES.items() if any(cue in cl for cue in cues)), None)
    if relation is None:
        return None
    ps = _match_titles(claim, cand_papers, k=2)
    ps_str = [str(x) for x in ps]
    if subject_id is not None and str(subject_id) in ps_str:
        others = [p for p in ps if str(p) != str(subject_id)]
        paper_a, paper_b = subject_id, (others[0] if others else None)
    elif len(ps) >= 2:
        paper_a, paper_b = ps[0], ps[1]
    elif len(ps) == 1:
        paper_a = subject_id if subject_id is not None else ps[0]
        paper_b = ps[0] if subject_id is not None else None
    else:
        paper_a = paper_b = None
    return {"relation": relation, "paper_a": paper_a, "paper_b": paper_b}

def _sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", str(text).strip()) if len(s.strip()) >= 15]

# ---- Rationale generation (OpenRouter, gated by RUN_LLM_GEN) ------------------
# Relation-eliciting prompt matching the strategy used to construct SciRecBench (see the
# manuscript's Generation Agent paragraph): the candidate this rationale is about is [1],
# a couple of the other top candidates are [2].. so the model can make (and we can then
# verify) an actual paper<->paper relational claim, grounded strictly in the abstracts,
# with an explicit honest out ("independent topics") instead of being pushed to invent one.
def _build_bracket_list(main_cand, other_cands):
    items = [main_cand] + list(other_cands)
    lines = [f'[{i}] "{p.get("title","")}" : {(p.get("abstract","") or "")[:300]}'
             for i, p in enumerate(items, 1)]
    return "\n".join(lines), items

def generate_rationale_llm(query, cand, other_cands):
    """Returns (rationale_text_or_None, bracket_items). bracket_items is the exact
    numbered list shown to the model ([1]=cand, [2..]=other_cands), needed so
    detect_relation's bracket-number resolution lines up with THIS call's numbering."""
    listing, items = _build_bracket_list(cand, other_cands)
    if not (RUN_LLM_GEN and OPENROUTER_API_KEY):
        return None, items
    prompt = (
        "You are a scientific paper recommendation assistant. Below are recommended papers "
        "for a user query, numbered [1]-[N]; [1] is the paper you are explaining.\n\n"
        f"Query: {query[:200]}\n\n{listing}\n\n"
        "In 2-4 sentences, explain why [1] is recommended, AND state how [1] relates to at "
        "least one other numbered paper above (e.g., builds on, uses the method of, compares "
        "with, or shares a topic with), if a genuine relationship is evident from the "
        "abstracts. Refer to papers by their bracket number, e.g. \"[1] builds on [2]\". "
        "Ground every claim STRICTLY in the abstracts above; if no real relationship is "
        "evident, say the papers address related but independent topics rather than "
        "inventing one."
    )
    body = {"model": GEN_MODEL, "messages": [{"role": "user", "content": prompt}],
            "max_tokens": 180, "temperature": 0.2,
            "provider": {"order": [OPENROUTER_PROVIDER], "allow_fallbacks": False}}
    try:
        r = requests.post(f"{OPENROUTER_BASE}/chat/completions",
                          headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}"},
                          json=body, timeout=60)
        r.raise_for_status()
        return r.json()["choices"][0]["message"]["content"].strip(), items
    except Exception as e:
        return f"[gen error: {e}]", items

print("Relation extraction + generation helpers defined.")

CiteCheck ready | graph cache: 1100 papers | S2: True
Relation extraction + generation helpers defined.


In [6]:
CITATION_TRUST_THRESHOLD = 0.80
TOP_K = 10

# Weighted hybridization (Burke, 2002) fusion weights for the retrieval agent -- tuned via
# nb10's validation sweep (a grid that included "dense alone" as a floor candidate). Graph
# is kept at a small nonzero weight rather than the pure-ranking-optimal 0.0 so citation
# centrality remains a real, functioning signal (matching this system's stated design), at
# a small, measured ranking-accuracy cost. Kept in sync with nb05's HybridRetrievalAgent
# defaults.
LAMBDA_DENSE, LAMBDA_BM25, LAMBDA_GRAPH, LAMBDA_CF = 0.6, 0.1, 0.05, 0.25


def node_profile_agent(state):
    """Node: User Profile Agent - fetch/init profile and contextualize the query."""
    user_id    = state["user_id"]
    query_type = state.get("query_type", "history_based")
    _raw = user_profiles_store.get(user_id)
    _cold = {"user_id": user_id, "profile_vector": [], "domain_keywords": [],
             "recent_query": state["query"], "interaction_count": 0,
             "recent_interest_shift": False}
    if isinstance(_raw, dict):
        profile = _raw
    elif isinstance(_raw, list):              # export stores the profile vector directly
        profile = dict(_cold, profile_vector=_raw)
    else:
        profile = _cold

    enriched_query = state["query"]
    if query_type in ("natural_language", "hybrid"):
        keywords = profile.get("domain_keywords", [])[:5]
        if keywords:
            if profile.get("recent_interest_shift"):
                enriched_query = f"{state['query']} (recent focus: {keywords[0]})"
            else:
                enriched_query = f"{state['query']}, especially related to {', '.join(keywords)}"
    profile["recent_query"] = enriched_query
    state["query"] = enriched_query
    state["user_profile"] = profile
    print(f"[ProfileAgent] User={user_id} | mode={query_type}")
    return state


def node_retrieval_agent(state):
    """Node: Hybrid Retrieval Agent - dense + BM25 + collaborative filtering (ALS) +
    citation-graph centrality, all four signals min-max normalized across the candidate
    set before blending (same fix as nb05's HybridRetrievalAgent -- raw dense cosine over
    this embedding space spans only ~0.76-0.87, so an unnormalized blend let 0-1-scaled
    BM25/graph/CF dominate regardless of nominal weight), then profile rerank."""
    query    = state["query"]
    profile  = state["user_profile"]
    excluded = set(state.get("low_trust_paper_ids", []))
    pv = profile.get("profile_vector", [])
    profile_vec = np.array(pv, dtype=np.float32) if pv else None

    # CF user lookup: als_user_ids.json's ids are ints (rated_papers.csv's user_id space);
    # state["user_id"] may be a different id space entirely (e.g. "test_user_001") -- try
    # to resolve, fall back to no-CF (cold-start) if it doesn't parse or isn't in the model.
    try:
        cf_uid = int(state["user_id"])
    except (TypeError, ValueError):
        cf_uid = None
    cf_urow = cf_user_row.get(cf_uid) if cf_uid is not None else None

    # Dense retrieval via direct cosine (matches nb05; encoder matches doc embeddings)
    q = embed_model.encode([QUERY_PREFIX + query], normalize_embeddings=True,
                           convert_to_numpy=True)[0].astype(np.float32)
    sims = (emb_matrix @ q) / _doc_norms
    k = min(TOP_K * 3, len(sims))
    top = np.argpartition(-sims, k - 1)[:k]
    top = top[np.argsort(-sims[top])]

    _bm_all = bm25_index.get_scores(query.lower().split())   # BM25 over corpus (hybrid signal)
    _bm_max = float(_bm_all.max()) or 1.0

    raw = []   # (pid, dense, bm25, graph, cf, sources)
    seen = set()
    for idx in top:
        pid = _idx_to_pid.get(int(idx), str(int(idx)))
        if pid in excluded or pid in seen:
            continue
        seen.add(pid)
        sim = float(sims[idx])
        bm  = float(_bm_all[_bm25_pos[pid]] / _bm_max) if pid in _bm25_pos else 0.0
        g   = float(graph_scores.get(str(pid), 0.0))
        cf  = 0.0
        if cf_urow is not None and cf_item_factors is not None:
            iidx = id_mapping.get(str(pid))
            if iidx is not None:
                cf = float(np.dot(cf_user_factors[cf_urow], cf_item_factors[iidx]))
        _src = ["dense"] + (["bm25"] if bm > 0 else [])
        raw.append((pid, sim, bm, g, cf, _src))

    def _minmax(vals):
        v = np.array(vals, dtype=np.float64)
        lo, hi = v.min(), v.max()
        return (v - lo) / (hi - lo) if hi > lo else np.zeros_like(v)

    n_dense = _minmax([r[1] for r in raw])
    n_bm25  = _minmax([r[2] for r in raw])
    n_graph = _minmax([r[3] for r in raw])
    n_cf    = _minmax([r[4] for r in raw])

    candidates = []
    for i, (pid, sim, bm, g, cf, _src) in enumerate(raw):
        final = (LAMBDA_DENSE * n_dense[i] + LAMBDA_BM25 * n_bm25[i] +
                 LAMBDA_GRAPH * n_graph[i] + LAMBDA_CF * n_cf[i])

        if profile_vec is not None and str(pid) in id_mapping:
            pe = emb_matrix[id_mapping[str(pid)]]
            ps = float(np.dot(profile_vec, pe) /
                       (np.linalg.norm(profile_vec) * np.linalg.norm(pe) + 1e-9))
            final = 0.7 * final + 0.3 * ps
        crag = "CORRECT" if sim > 0.75 else ("AMBIGUOUS" if sim > 0.35 else "INCORRECT")

        title, abstract, year = "", "", None
        if pid in _pdf.index:
            row = _pdf.loc[pid]
            if "title" in row:    title = str(row["title"])
            if "abstract" in row: abstract = str(row["abstract"])[:400]
            for c in ("year", "published", "date"):
                if c in row and pd.notna(row[c]):
                    m = re.search(r"(\d{4})", str(row[c]))
                    if m:
                        year = int(m.group(1)); break
        candidates.append({"paper_id": pid, "title": title, "abstract": abstract, "year": year,
                           "dense_score": sim, "bm25_score": bm, "graph_score": g, "cf_score": cf,
                           "final_score": final, "crag_label": crag, "sources": _src})

    candidates.sort(key=lambda c: c["final_score"], reverse=True)
    candidates = [c for c in candidates if c["crag_label"] != "INCORRECT"][:TOP_K]
    state["candidates"] = candidates
    print(f"[RetrievalAgent] {len(candidates)} candidates (iter {state.get('retrieval_iteration', 0)})")
    return state


def node_critic_generator(state):
    """Node: Critic-Generator (the graph-grounded Checker).
    Stages 1-3 = text verification (CRAG / Self-RAG overlap / CoVe existence);
    Stage 4 = CiteCheck graph verification of relational (A, relation, B) claims.

    Measurement vs. behavior, kept deliberately separate for the ablation's sake:
    Stage 4 (detect_relation + citecheck.verify_relation) is run for EVERY config
    ("a"/"b"/"c" alike) -- it is a free, local citation-graph lookup, not an LLM call.
    What DIFFERS by config is whether a confirmed relational hallucination can EXCLUDE
    a candidate: only for "c" (see `excluded` below). This lets the ablation measure
    the TRUE relational-hallucination content of what "a"/"b" produce (their own
    decisions never see graph info) while scoring all three configs with the same
    uniform yardstick.

    BUG FIXED (found via a free RUN_LLM_GEN=False dry run before spending any real
    generation credit -- exactly why that dry run is worth doing first): the first cut
    of this fix multiplied trust*(1-risk) and gated ALL THREE configs on that product at
    the ORIGINAL 0.80 threshold. Stage 2's risk is a crude query/abstract word-overlap
    proxy that rarely dips below ~0.11 for realistic text, so trust=0.9 candidates were
    failing the gate from Stage-2 noise ALONE, in configs "a"/"b" where nothing should
    have been excluded at all (they never see graph info) -- collapsing final
    recommendations to ~0 across every config, not just "c". The gate below now excludes
    for two INDEPENDENT reasons instead of one multiplied score: (1) trust itself below
    threshold (the ORIGINAL, pre-existing criterion, unaffected by risk), or (2) a
    citation-graph-CONFIRMED relational hallucination on THIS candidate, and only when
    config "c" is allowed to act on it. Stage 2's risk score still feeds the SORT ORDER
    (as it always did) but no longer contaminates the inclusion decision."""
    query      = state["query"]
    candidates = state["candidates"]
    keywords   = state["user_profile"].get("domain_keywords", [])[:5]
    config     = state.get("verification_config", "c")
    acts_on_citecheck = (config == "c")

    # Build the focused citation graph over this query's candidates (one S2 batch,
    # cached) -- always, for measurement, regardless of config (see docstring).
    citecheck.ensure([c["paper_id"] for c in candidates])

    verification_results = []
    n_rel_halluc = 0
    for cand in candidates:
        abstract = cand.get("abstract", "")
        dense_sc = cand["dense_score"]
        rel_verdicts = []

        if dense_sc < 0.35:                                   # Stage 1 CRAG: discard
            risk, trust, rationale = 1.0, 0.0, "[FILTERED: low retrieval confidence]"
        else:
            # Show the model a couple of the OTHER top candidates too (relation-eliciting
            # prompt, matches SciRecBench's construction) so it can make (and CiteCheck can
            # then verify) an actual paper<->paper relational claim, not just single-paper
            # relevance. other_for_gen order fixes the bracket numbering match_pool relies on.
            other_for_gen = [c for c in candidates if c["paper_id"] != cand["paper_id"]][:2]
            gen, match_pool = generate_rationale_llm(query, cand, other_for_gen)
            if gen:
                rationale = gen
            else:
                # Deterministic stub asserting a relation to another candidate, so the
                # GRAPH Checker is exercised end-to-end for free (RUN_LLM_GEN=False).
                other = next((c for c in candidates if c["paper_id"] != cand["paper_id"]), None)
                if other and other.get("title"):
                    rationale = (f"This paper is relevant to '{query[:60]}'. "
                                 f"It builds on \"{other['title']}\".")
                else:
                    rationale = f"Relevant to '{query[:60]}' - topics: {', '.join(keywords[:3])}"
                match_pool = candidates   # stub quotes a title, not a bracket number

            # Stage 2 (text): query-abstract overlap proxy for atomic groundedness.
            # NOTE: feeds sort order only now, not the inclusion gate -- see docstring.
            qw = set(query.lower().split()); aw = set(abstract.lower().split())
            overlap = len(qw & aw) / max(len(qw), 1)
            risk = max(0.0, 1.0 - min(overlap * 3, 1.0))
            # Stage 3 (text CoVe): corpus existence
            trust = 0.9 if str(cand["paper_id"]) in id_mapping else 0.65

            # Stage 4 (GRAPH - CiteCheck): verify relational claims in the rationale.
            # Always computed (measurement); only ALLOWED TO EXCLUDE when acts_on_citecheck
            # (config "c") -- see this function's docstring.
            # subject_id=cand["paper_id"] lets detect_relation correctly resolve the
            # implicit self-reference ("this paper..." / "[1]" when [1] IS the subject)
            # instead of losing paper_b to a None default (see detect_relation's docstring).
            for sent in _sentences(rationale):
                det = detect_relation(sent, match_pool, subject_id=cand["paper_id"])
                if not det:
                    continue
                a = det["paper_a"]
                b = det["paper_b"]
                verdict = citecheck.verify_relation(str(a), det["relation"], str(b)) if (a and b) else "unverifiable"
                rel_verdicts.append({"claim": sent[:160], "relation": det["relation"],
                                     "paper_a": (str(a) if a else None), "paper_b": (str(b) if b else None),
                                     "verdict": verdict})
                if verdict == "unsupported":                  # relational hallucination
                    n_rel_halluc += 1
                    if acts_on_citecheck:
                        risk = max(risk, 0.85)                 # pushes this candidate down in sort order

        n_rel_halluc_this_cand = sum(1 for r in rel_verdicts if r["verdict"] == "unsupported")
        # Exclusion gate: two INDEPENDENT reasons, not a multiplied score (see docstring
        # for why the multiplied version was a bug). Stage 2's risk never excludes by
        # itself; only (1) low corpus-existence trust, or (2) a config-"c" CONFIRMED
        # relational hallucination on THIS candidate.
        excluded_flag = (trust < CITATION_TRUST_THRESHOLD) or (acts_on_citecheck and n_rel_halluc_this_cand > 0)
        sort_score = trust * (1 - risk)   # sort order only; does not affect inclusion

        verification_results.append({
            "paper_id": cand["paper_id"], "title": cand.get("title", ""), "rationale": rationale,
            "hallucination_risk_score": risk, "citation_trust_score": trust,
            "sort_score": sort_score, "excluded": excluded_flag,
            "final_score": cand["final_score"], "relational_verdicts": rel_verdicts,
            "n_relational_hallucinations": n_rel_halluc_this_cand,
        })

    verification_results.sort(key=lambda r: r["sort_score"], reverse=True)
    state["verification_results"] = verification_results
    state["low_trust_paper_ids"]  = [r["paper_id"] for r in verification_results if r["excluded"]]
    state["n_relational_hallucinations"] = n_rel_halluc
    print(f"[CriticGenerator] config={config} | verified {len(verification_results)} | "
          f"relational hallucinations detected: {n_rel_halluc}"
          f"{' (acted on)' if acts_on_citecheck else ' (measured only, not acted on)'}")
    return state


def node_orchestrator_finalize(state):
    """Node: Orchestrator - compile final recommendations (trust-gated; a candidate is
    kept unless `excluded` -- see node_critic_generator's docstring for the two
    independent exclusion reasons and the bug that using a multiplied score caused)."""
    vrs = state.get("verification_results", [])
    final = [{
        "rank": i + 1, "paper_id": vr["paper_id"], "title": vr["title"], "rationale": vr["rationale"],
        "hallucination_risk": round(vr["hallucination_risk_score"], 3),
        "citation_trust": round(vr["citation_trust_score"], 3),
        "relational_hallucinations": vr.get("n_relational_hallucinations", 0),
        "relational_verdicts": vr.get("relational_verdicts", []),
        "overall_score": round(vr["final_score"] * vr["sort_score"], 3),
    } for i, vr in enumerate(vrs) if not vr["excluded"]][:TOP_K]
    state["final_recommendations"] = final
    state["status"] = "complete"
    print(f"[Orchestrator] final {len(final)} papers | "
          f"{sum(r['relational_hallucinations'] for r in final)} relational hallucinations in kept set")
    return state


print("Agent node functions defined (Checker = text Stages 1-3 + graph CiteCheck Stage 4; "
      "Retrieval = dense + BM25 + CF + graph, shortlist-normalized fusion).")

Agent node functions defined (Checker = text Stages 1-3 + graph CiteCheck Stage 4; Retrieval = dense + BM25 + CF + graph, shortlist-normalized fusion).


## 3. Routing Logic (Conditional Edges)

In [7]:
def route_after_critic(
    state: MAPERState,
) -> Literal["re_retrieve", "finalize"]:
    """
    Config "a" (single-pass, text-only) never re-routes -- that's the definition of
    single-pass, representative of prior single-pass verified-RAG recommenders (see
    the manuscript's Experimental Setup). Configs "b"/"c" use the same re-routing rule:
    if too many papers are excluded (see node_critic_generator's `excluded` field) AND
    we haven't exceeded max iterations, re-route to retrieval for more candidates.
    Otherwise finalize.
    """
    if state.get("verification_config", "c") == "a":
        return "finalize"

    vrs       = state.get("verification_results", [])
    iteration = state.get("retrieval_iteration", 0)
    max_iter  = state.get("max_iterations", 2)

    high_trust_count = len([v for v in vrs if not v["excluded"]])

    if high_trust_count < TOP_K // 2 and iteration < max_iter:
        print(f"[Router] Only {high_trust_count} high-trust papers — re-retrieving (iter {iteration + 1})")
        return "re_retrieve"
    return "finalize"


def increment_iteration(state: MAPERState) -> MAPERState:
    """Increment retrieval iteration counter before re-retrieval."""
    state["retrieval_iteration"] = state.get("retrieval_iteration", 0) + 1
    return state


print("Routing logic defined.")

Routing logic defined.


## 4. Build LangGraph StateGraph

In [8]:
if LANGGRAPH_AVAILABLE:
    from langgraph.graph import StateGraph, END

    # Build the graph
    builder = StateGraph(MAPERState)

    # Add nodes
    builder.add_node("profile_agent",    node_profile_agent)
    builder.add_node("retrieval_agent",  node_retrieval_agent)
    builder.add_node("critic_generator", node_critic_generator)
    builder.add_node("increment_iter",   increment_iteration)
    builder.add_node("finalize",         node_orchestrator_finalize)

    # Define edges
    builder.set_entry_point("profile_agent")
    builder.add_edge("profile_agent",    "retrieval_agent")
    builder.add_edge("retrieval_agent",  "critic_generator")
    builder.add_conditional_edges(
        "critic_generator",
        route_after_critic,
        {
            "re_retrieve": "increment_iter",
            "finalize":    "finalize",
        },
    )
    builder.add_edge("increment_iter",   "retrieval_agent")  # loop back
    builder.add_edge("finalize",         END)

    maper_graph = builder.compile()
    print("MAPER LangGraph compiled successfully.")

else:
    # Fallback: sequential pipeline without LangGraph
    def maper_pipeline(user_id: str, query: str,
                       query_type: str = "history_based",
                       verification_config: str = "c") -> MAPERState:
        state: MAPERState = {
            "user_id": user_id,
            "query": query,
            "query_type": query_type,
            "original_query": query,
            "verification_config": verification_config,
            "user_profile": {},
            "candidates": [],
            "verification_results": [],
            "final_recommendations": [],
            "retrieval_iteration": 0,
            "max_iterations": 2,
            "low_trust_paper_ids": [],
            "n_relational_hallucinations": 0,
            "status": "running",
            "error_message": "",
        }
        state = node_profile_agent(state)
        state = node_retrieval_agent(state)
        state = node_critic_generator(state)
        if verification_config != "a" and route_after_critic(state) == "re_retrieve":
            state = increment_iteration(state)
            state = node_retrieval_agent(state)
            state = node_critic_generator(state)
        state = node_orchestrator_finalize(state)
        return state
    print("LangGraph not available — using sequential pipeline fallback.")


MAPER LangGraph compiled successfully.


## 5. End-to-End Inference Test

In [9]:
def make_initial_state(user_id: str, query: str,
                       query_type: str = "history_based",
                       max_iterations: int = 2,
                       verification_config: str = "c") -> MAPERState:
    """Helper to build a clean initial MAPERState for any query mode.
    verification_config: "a" (single-pass, text-only), "b" (multi-stage, text-only),
    or "c" (multi-stage + CiteCheck, the paper's full system). Defaults to "c" so
    existing calls in this notebook keep their prior behavior unchanged."""
    return {
        "user_id": user_id,
        "query": query,
        "query_type": query_type,
        "original_query": query,
        "verification_config": verification_config,
        "user_profile": {},
        "candidates": [],
        "verification_results": [],
        "final_recommendations": [],
        "retrieval_iteration": 0,
        "max_iterations": max_iterations,
        "low_trust_paper_ids": [],
        "n_relational_hallucinations": 0,
        "status": "running",
        "error_message": "",
    }


# ── Single inference test (natural_language mode) ─────────────────────────────
test_user_id = list(user_profiles_store.keys())[0] if user_profiles_store else "test_user_001"
test_query   = "multi-agent systems for hallucination-free scientific paper recommendation"

initial_state = make_initial_state(test_user_id, test_query, query_type="natural_language")

print(f"Query      : {test_query}")
print(f"User       : {test_user_id}")
print(f"Query mode : {initial_state['query_type']}")
print("-" * 60)

if LANGGRAPH_AVAILABLE:
    final_state = maper_graph.invoke(initial_state)
else:
    final_state = maper_pipeline(test_user_id, test_query, query_type="natural_language")

print(f"\nStatus: {final_state['status']}")
print(f"Final recommendations: {len(final_state['final_recommendations'])}")


Query      : multi-agent systems for hallucination-free scientific paper recommendation
User       : 0
Query mode : natural_language
------------------------------------------------------------
[ProfileAgent] User=0 | mode=natural_language
[RetrievalAgent] 10 candidates (iter 0)
[CriticGenerator] config=c | verified 10 | relational hallucinations detected: 0 (acted on)
[Orchestrator] final 10 papers | 0 relational hallucinations in kept set

Status: complete
Final recommendations: 10


In [10]:
# ── Display final recommendations ─────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"MAPER Recommendations for: \"{test_query}\"")
print(f"{'='*70}")
for rec in final_state["final_recommendations"]:
    print(f"\n#{rec['rank']} — {rec['title'][:60]}")
    print(f"    Hallucination Risk : {rec['hallucination_risk']:.3f}")
    print(f"    Citation Trust     : {rec['citation_trust']:.3f}")
    print(f"    Overall Score      : {rec['overall_score']:.3f}")
    print(f"    Relational halluc. : {rec.get('relational_hallucinations', 0)}")
    print(f"    Rationale          : {rec['rationale'][:150]}...")


MAPER Recommendations for: "multi-agent systems for hallucination-free scientific paper recommendation"

#1 — Seeing Through the Fog: A Cost-Effectiveness Analysis of Hal
    Hallucination Risk : 0.400
    Citation Trust     : 0.900
    Overall Score      : 0.311
    Relational halluc. : 0
    Rationale          : [gen error: 429 Client Error: Too Many Requests for url: https://openrouter.ai/api/v1/chat/completions]...

#2 — Good Parenting is all you need -- Multi-agentic LLM Hallucin
    Hallucination Risk : 0.600
    Citation Trust     : 0.900
    Overall Score      : 0.294
    Relational halluc. : 0
    Rationale          : [gen error: 'choices']...

#3 — Small Agent Can Also Rock! Empowering Small Language Models 
    Hallucination Risk : 0.600
    Citation Trust     : 0.900
    Overall Score      : 0.231
    Relational halluc. : 0
    Rationale          : [gen error: 'choices']...

#4 — SLPL SHROOM at SemEval2024 Task 06: A comprehensive study on
    Hallucination Risk : 0.600
  

## 5b. All Three Query Modes Demo


In [11]:
demo_user  = list(user_profiles_store.keys())[0] if user_profiles_store else "demo_user"
demo_query = "papers on hallucination detection using retrieval augmented generation"

modes = ["history_based", "natural_language", "hybrid"]
mode_results = {}

print("=" * 70)
print("MAPER — Three Query Mode Comparison")
print("=" * 70)

for mode in modes:
    print(f"\n{'─'*70}")
    print(f"Mode: {mode}")
    print(f"{'─'*70}")
    s = make_initial_state(demo_user, demo_query, query_type=mode, max_iterations=1)

    if LANGGRAPH_AVAILABLE:
        result = maper_graph.invoke(s)
    else:
        result = maper_pipeline(demo_user, demo_query, query_type=mode)

    mode_results[mode] = result
    recs = result["final_recommendations"]
    print(f"  Enriched query : {result['query'][:90]}")
    print(f"  Recommendations: {len(recs)}")
    if recs:
        top = recs[0]
        print(f"  Top paper      : {top['title'][:60]}")
        print(f"  Trust / Risk   : {top['citation_trust']} / {top['hallucination_risk']}")

print("\n✓ All three modes completed.")


MAPER — Three Query Mode Comparison

──────────────────────────────────────────────────────────────────────
Mode: history_based
──────────────────────────────────────────────────────────────────────
[ProfileAgent] User=0 | mode=history_based
[RetrievalAgent] 10 candidates (iter 0)
[CriticGenerator] config=c | verified 10 | relational hallucinations detected: 1 (acted on)
[Orchestrator] final 9 papers | 0 relational hallucinations in kept set
  Enriched query : papers on hallucination detection using retrieval augmented generation
  Recommendations: 9
  Top paper      : Lynx: An Open Source Hallucination Evaluation Model
  Trust / Risk   : 0.9 / 0.0

──────────────────────────────────────────────────────────────────────
Mode: natural_language
──────────────────────────────────────────────────────────────────────
[ProfileAgent] User=0 | mode=natural_language
[RetrievalAgent] 10 candidates (iter 0)
[CriticGenerator] config=c | verified 10 | relational hallucinations detected: 2 (acted on)

## 6. Verification-architecture ablation (Table `tab:ablation`)

Runs the three configurations described in the manuscript's Experimental Setup --
**(a)** single-pass, text-only; **(b)** multi-stage, text-only; **(c)** multi-stage +
CiteCheck (ours) -- on the SAME queries, taken from the 68-instance human-verified
checkpoint (`scireqa_human_verified_subset.jsonl`), not an arbitrary slice of the full
10k-instance silver benchmark.

**Scoring, and why it's not circular:** all three configs' Stage-4 CiteCheck check now
always runs (see `node_critic_generator`'s docstring) -- it's a free local graph lookup,
not an LLM call -- but only config "c" lets it influence the pipeline's own behavior
(risk score / exclusion / re-routing). This means the relational-hallucination rate
reported below is the SAME external, uniform yardstick applied to whatever each config
actually produced and surfaced, not each config grading its own homework.

**What this is NOT:** it is not scored against fresh human annotation of the newly
generated rationales -- the existing human gold was annotated on the ORIGINAL static
benchmark's rationales, which is different text than whatever this live run generates.
The number below is an automatic, citation-graph-verified proxy for the hallucination
rate, with the same honestly-disclosed precision/coverage limits established in
`02_relational_finding.ipynb`. Treat it as what it is -- a repeatable, disclosed proxy
metric -- and consider a manual spot-check of a sample of flagged claims before quoting
it as a headline number.

**Cost note:** with `RUN_LLM_GEN=False` (this notebook's default), all three configs
use the same deterministic stub rationale, which will show whether the ARCHITECTURE
(re-routing, exclusion-on-flag) behaves as intended, but is NOT a meaningful test of
real generation differences across configs. Set `RUN_LLM_GEN=True` in cell 1 for a
real result -- this spends OpenRouter credit (roughly instances x configs x candidates
generation calls; see the printed running count below before committing to the full
68-instance run).

In [12]:
from tqdm import tqdm
from collections import Counter

# Scope to the 68-instance HUMAN-VERIFIED checkpoint, not an arbitrary slice of the
# full 10k silver benchmark -- this is the set the manuscript's other numbers are
# computed on, so the ablation should run on the same instances/queries.
gold_path = _find("scireqa_human_verified_subset.jsonl")
assert gold_path, "attach scireqa_human_verified_subset.jsonl (from agentic-nb2) to run the ablation"
gold_instances = [json.loads(l) for l in open(gold_path, encoding="utf-8") if l.strip()]
print(f"Gold instances: {len(gold_instances)} (from {gold_path})")

# Raise this once you've confirmed cost/behavior on a small pilot (the 5-instance dry
# run and pilot already validated the architecture and cost: ~$0.0047/instance).
ABLATION_LIMIT = 68
CONFIGS = ["a", "b", "c"]

# ── Checkpointing (RESUME SUPPORT) ────────────────────────────────────────────
# The earlier version of this cell only wrote results to disk after the ENTIRE loop
# finished -- on a multi-hour paid run, a Kaggle disconnect partway through would lose
# every generation call made so far with nothing to show for it. Now every (instance,
# config) result is saved to OUT_DIR immediately after it completes, keyed by instance
# id, and reloaded on start -- already-completed (instance, config) pairs are skipped
# entirely (no re-spend, no re-run), so an interrupted session just needs "Run All"
# again to pick up where it left off.
CHECKPOINT_PATH = OUT_DIR / "ablation_checkpoint.json"

def _load_checkpoint():
    if CHECKPOINT_PATH.exists():
        try:
            return json.loads(CHECKPOINT_PATH.read_text(encoding="utf-8"))
        except Exception:
            return {}
    return {}

def _save_checkpoint(ckpt):
    CHECKPOINT_PATH.write_text(json.dumps(ckpt), encoding="utf-8")

checkpoint = _load_checkpoint()
if checkpoint:
    _done_pairs = sum(len(v) for v in checkpoint.values())
    print(f"Resuming from checkpoint: {len(checkpoint)} instances, {_done_pairs} (instance,config) "
          f"results already completed -> {CHECKPOINT_PATH}")
else:
    print("No checkpoint found -- starting fresh.")

ablation_post = {cfg: [] for cfg in CONFIGS}
ablation_pre  = {cfg: [] for cfg in CONFIGS}
n_final_recs  = {cfg: [] for cfg in CONFIGS}   # len(final_recommendations) per instance
n_gen_calls = 0

# Seed the in-memory aggregates from whatever the checkpoint already has.
for _iid, _per_cfg in checkpoint.items():
    for cfg in CONFIGS:
        if cfg in _per_cfg:
            ablation_post[cfg].append(_per_cfg[cfg]["post"])
            ablation_pre[cfg].append(_per_cfg[cfg]["pre"])
            n_final_recs[cfg].append(_per_cfg[cfg]["n_final"])

for idx, inst in enumerate(tqdm(gold_instances[:ABLATION_LIMIT], desc="Ablation (a/b/c)")):
    iid = inst.get("id") or f"idx{idx}"
    if iid in checkpoint and all(cfg in checkpoint[iid] for cfg in CONFIGS):
        continue   # this instance is fully done (all 3 configs) -- skip, no re-spend
    uid   = str(inst.get("user_id", "eval_user"))
    query = inst.get("query") or inst.get("original_query") or ""
    qtype = inst.get("query_type", "history_based")
    if not query:
        continue
    inst_record = checkpoint.get(iid, {})
    for cfg in CONFIGS:
        if cfg in inst_record:
            continue   # this specific config already done for this instance -- skip
        s = make_initial_state(uid, query, query_type=qtype, max_iterations=1, verification_config=cfg)
        rs = maper_graph.invoke(s) if LANGGRAPH_AVAILABLE else maper_pipeline(uid, query, query_type=qtype, verification_config=cfg)

        post_verds = dict(Counter(v["verdict"] for r in rs["final_recommendations"]
                                  for v in r.get("relational_verdicts", [])))
        # verification_results reflects the LAST iteration only (it's overwritten each
        # time the Critic-Generator runs) -- for "a" that's the only iteration anyway;
        # for "b"/"c" it's whatever candidates the final re-routing round considered.
        pre_verds = dict(Counter(v["verdict"] for r in rs.get("verification_results", [])
                                 for v in r.get("relational_verdicts", [])))
        n_final = len(rs["final_recommendations"])

        ablation_post[cfg].append(post_verds)
        ablation_pre[cfg].append(pre_verds)
        n_final_recs[cfg].append(n_final)
        n_gen_calls += len(rs.get("candidates", []))

        inst_record[cfg] = {"post": post_verds, "pre": pre_verds, "n_final": n_final}
        checkpoint[iid] = inst_record
        _save_checkpoint(checkpoint)   # persist after EVERY (instance, config) result

print(f"\nTotal candidate-generation calls made THIS RUN across all configs: {n_gen_calls} "
      f"(each is an OpenRouter call if RUN_LLM_GEN=True, else free stub; checkpoint-skipped "
      f"pairs from a prior run are not recounted here)")

def _summarize(raw_by_cfg):
    out = {}
    for cfg in CONFIGS:
        total = Counter()
        for c in raw_by_cfg[cfg]:
            total.update(c)
        adjudicable = total.get("supported", 0) + total.get("unsupported", 0)
        rate = total.get("unsupported", 0) / adjudicable if adjudicable else float("nan")
        out[cfg] = {"verdict_counts": dict(total), "adjudicable": adjudicable,
                    "relational_hallucination_rate": rate}
    return out

summary_post = _summarize(ablation_post)
summary_pre  = _summarize(ablation_pre)

CONFIG_LABELS = {
    "a": "(a) Single-pass, text-only",
    "b": "(b) Multi-stage, text-only",
    "c": "(c) Multi-stage + CiteCheck (ours)",
}

def _print_table(title, summary):
    print(f"\n{title}")
    print(f"{'Configuration':38s} {'Rate':>8s} {'Adjudicable':>12s}  Verdict counts")
    for cfg in CONFIGS:
        s = summary[cfg]
        rate_str = f"{s['relational_hallucination_rate']:.1%}" if s["adjudicable"] else "n/a"
        print(f"{CONFIG_LABELS[cfg]:38s} {rate_str:>8s} {s['adjudicable']:>12d}  {s['verdict_counts']}")

_print_table("=== PRE-exclusion (everything the last pass considered; comparable across configs) ===", summary_pre)
_print_table("=== POST-exclusion (what's actually surfaced; circular for config c read alone -- see yield) ===", summary_post)

print(f"\n{'Configuration':38s} {'Avg recs':>10s} {'Zero-rec instances':>20s}")
yield_stats = {}
for cfg in CONFIGS:
    counts = n_final_recs[cfg]
    avg = sum(counts) / len(counts) if counts else float("nan")
    n_zero = sum(1 for c in counts if c == 0)
    yield_stats[cfg] = {"avg_final_recs": avg, "n_zero_rec_instances": n_zero, "n_instances": len(counts)}
    print(f"{CONFIG_LABELS[cfg]:38s} {avg:>10.2f} {n_zero:>12d} / {len(counts)}")

with open(OUT_DIR / "ablation_results.json", "w", encoding="utf-8") as f:
    json.dump({"n_instances": len(checkpoint),
               "summary_pre_exclusion": summary_pre, "summary_post_exclusion": summary_post,
               "yield_stats": yield_stats, "run_llm_gen": RUN_LLM_GEN}, f, indent=2)
print(f"\nSaved: {OUT_DIR / 'ablation_results.json'}")
print(f"Checkpoint ({len(checkpoint)} instances done): {CHECKPOINT_PATH}")
print("\nNOTE on reading these numbers:")
print("- PRE-exclusion is the closer analogue to 'relational-hallucination rate of the")
print("  resulting rationales' the manuscript asks for -- it isn't affected by config c's")
print("  own filtering, so a/b/c are on comparable footing (modulo re-routing changing which")
print("  candidates config c's last pass considered).")
print("- POST-exclusion's ~0% for config c is EXPECTED BY CONSTRUCTION (the exclusion rule")
print("  IS the same check), not evidence of better generation -- read it together with the")
print("  yield table: a low/zero hallucination rate that comes with a lower average recs")
print("  is a precision/coverage trade-off, not a free win. Report both.")
print("- This is an automatic, citation-graph-verified proxy metric, not scored against fresh")
print("  human annotation of these specific generated rationales.")
print("- If RUN_LLM_GEN=False, all three configs saw the same deterministic stub rationale;")
print("  this run only validates the ARCHITECTURE (routing / exclusion-on-flag), not real")
print("  generation differences.")

Gold instances: 68 (from /kaggle/input/datasets/joytuntonny/rationale-subset/scireqa_human_verified_subset.jsonl)
No checkpoint found -- starting fresh.


Ablation (a/b/c):   0%|          | 0/68 [00:00<?, ?it/s]

[ProfileAgent] User=20826 | mode=hybrid
[RetrievalAgent] 10 candidates (iter 0)
[CriticGenerator] config=a | verified 10 | relational hallucinations detected: 4 (measured only, not acted on)
[Orchestrator] final 10 papers | 4 relational hallucinations in kept set
[ProfileAgent] User=20826 | mode=hybrid
[RetrievalAgent] 10 candidates (iter 0)
[CriticGenerator] config=b | verified 10 | relational hallucinations detected: 5 (measured only, not acted on)
[Orchestrator] final 10 papers | 5 relational hallucinations in kept set
[ProfileAgent] User=20826 | mode=hybrid
[RetrievalAgent] 10 candidates (iter 0)
[CriticGenerator] config=c | verified 10 | relational hallucinations detected: 6 (acted on)
[Router] Only 4 high-trust papers — re-retrieving (iter 1)
[RetrievalAgent] 10 candidates (iter 1)


Ablation (a/b/c):   1%|▏         | 1/68 [03:11<3:33:46, 191.43s/it]

[CriticGenerator] config=c | verified 10 | relational hallucinations detected: 4 (acted on)
[Orchestrator] final 6 papers | 0 relational hallucinations in kept set
[ProfileAgent] User=7071 | mode=hybrid
[RetrievalAgent] 10 candidates (iter 0)
[CriticGenerator] config=a | verified 10 | relational hallucinations detected: 1 (measured only, not acted on)
[Orchestrator] final 10 papers | 1 relational hallucinations in kept set
[ProfileAgent] User=7071 | mode=hybrid
[RetrievalAgent] 10 candidates (iter 0)
[CriticGenerator] config=b | verified 10 | relational hallucinations detected: 0 (measured only, not acted on)
[Orchestrator] final 10 papers | 0 relational hallucinations in kept set
[ProfileAgent] User=7071 | mode=hybrid
[RetrievalAgent] 10 candidates (iter 0)


Ablation (a/b/c):   3%|▎         | 2/68 [06:01<3:16:58, 179.06s/it]

... (129 outputs left out)

Ablation (a/b/c):  99%|█████████▊| 67/68 [5:01:40<03:12, 192.04s/it]

[CriticGenerator] config=c | verified 10 | relational hallucinations detected: 12 (acted on)
[Orchestrator] final 1 papers | 0 relational hallucinations in kept set
[ProfileAgent] User=pm_cluster_220_user_2090 | mode=natural_language
[RetrievalAgent] 10 candidates (iter 0)
[CriticGenerator] config=a | verified 10 | relational hallucinations detected: 0 (measured only, not acted on)
[Orchestrator] final 10 papers | 0 relational hallucinations in kept set
[ProfileAgent] User=pm_cluster_220_user_2090 | mode=natural_language
[RetrievalAgent] 10 candidates (iter 0)
[CriticGenerator] config=b | verified 10 | relational hallucinations detected: 0 (measured only, not acted on)
[Orchestrator] final 10 papers | 0 relational hallucinations in kept set
[ProfileAgent] User=pm_cluster_220_user_2090 | mode=natural_language
[RetrievalAgent] 10 candidates (iter 0)


Ablation (a/b/c): 100%|██████████| 68/68 [5:03:57<00:00, 268.19s/it]

[CriticGenerator] config=c | verified 10 | relational hallucinations detected: 1 (acted on)
[Orchestrator] final 9 papers | 0 relational hallucinations in kept set

Total candidate-generation calls made THIS RUN across all configs: 2040 (each is an OpenRouter call if RUN_LLM_GEN=True, else free stub; checkpoint-skipped pairs from a prior run are not recounted here)

=== PRE-exclusion (everything the last pass considered; comparable across configs) ===
Configuration                              Rate  Adjudicable  Verdict counts
(a) Single-pass, text-only                48.4%          665  {'supported': 343, 'unsupported': 322, 'unverifiable': 225}
(b) Multi-stage, text-only                48.3%          666  {'supported': 344, 'unsupported': 322, 'unverifiable': 227}
(c) Multi-stage + CiteCheck (ours)        34.7%          585  {'supported': 382, 'unverifiable': 310, 'unsupported': 203}

=== POST-exclusion (what's actually surfaced; circular for config c read alone -- see yield) ===
Con

## 7. Summary

| Component | Technology | Role |
|---|---|---|
| Orchestrator | LangGraph StateGraph | Routing + state management |
| User Profile Agent | Hierarchical memory (nb04) | Dynamic interest modeling |
| Hybrid Retrieval Agent | BM25 + dense cosine + collaborative filtering (ALS, shared model from nb10) + graph centrality, shortlist-normalized (Burke, 2002 weighted hybridization) | Multi-source retrieval |
| **Checker — text** | CRAG + Self-RAG + CoVe (nb06) | Atomic groundedness + citation existence |
| **Checker — graph** | **CiteCheck `verify_relation` (nb01)** | **Relational (A,rel,B) verification — the blind spot text verifiers miss** |
| Re-routing | Conditional edges | Re-retrieve if too many low-trust papers |

The Checker is **graph-grounded**: Stages 1-3 verify atomic/text claims, **Stage 4 (CiteCheck)**
verifies relational claims against the citation graph and flags `unsupported` relations as
relational hallucinations. This is the option-2 composition (text-critic + CiteCheck) of the paper.

**Next**: Notebook 08 — DPO mitigation / Notebook 09 — Evaluation & Ablation.